In [3]:
import json, math, h5py, numpy as np, torch
from torch.utils.data import Dataset
from typing import List, Tuple

In [12]:
with h5py.File("../data/muon_data_7224_7247.h5", "r") as f:
    print(f.keys())
    print(f["hits"])
    print(f["hits"][0][0])

<KeysViewHDF5 ['energies', 'hits', 'mc_muons', 'rec_muons']>
<HDF5 dataset "hits": shape (21555, 512, 8), type "<f4">
[ 8.22491e+07  4.64465e+02  5.64880e+02  3.68470e+01  3.26855e-01
  8.97781e-01 -2.95209e-01  2.40000e+01]


In [ ]:
import json, math, h5py, numpy as np, torch
from torch.utils.data import Dataset
from typing import List, Tuple

def _event_offsets(paths: List[str], ds: str="hits"):
    sizes, offs = [], []
    total = 0
    for p in paths:
        with h5py.File(p, "r") as f:
            n = f[ds].shape[0]
        sizes.append(n)
        offs.append(total)
        total += n
    return sizes, offs, total

class VoxelizedTimeBinnedDataset(Dataset):
    """
    Returns (x, y) where:
      x: float tensor [T, C=2, nx, ny, nz] with channels {occ, count}
      y: int label ∈ {0,1,2}
    """
    def __init__(self, h5_paths: List[str], grid_json: str, hits_ds="hits",
                 label_group="mc_muons", label_col=0, dtype=np.float32):
        self.paths = list(h5_paths)
        self.hits_ds = hits_ds
        self.label_group = label_group
        self.label_col = label_col
        self.dtype = dtype

        with open(grid_json, "r") as f:
            G = json.load(f)
        self.cols = G["columns"]
        S = G["spatial"];  Tm = G["temporal"]

        self.xmin, self.xmax = S["x"]["min"], S["x"]["max"]
        self.ymin, self.ymax = S["y"]["min"], S["y"]["max"]
        self.zmin, self.zmax = S["z"]["min"], S["z"]["max"]
        self.nx = int(S["x"]["nx"]); self.ny = int(S["y"]["ny"]); self.nz = int(S["z"]["nz"])
        self.vx = float(S["x"]["voxsize"]); self.vy = float(S["y"]["voxsize"]); self.vz = float(S["z"]["voxsize"])

        self.tmin, self.tmax = Tm["t_min"], Tm["t_max"]
        self.dt = float(Tm["dt"])
        self.T = int(Tm["T"])  # 84

        # column indices (chronological mapping requested)
        self.tcol = int(self.cols["time"])
        self.xcol = int(self.cols["x"])
        self.ycol = int(self.cols["y"])
        self.zcol = int(self.cols["z"])

        self.sizes, self.offs, self.total = _event_offsets(self.paths, self.hits_ds)

    def __len__(self):
        return self.total

    def _locate(self, idx: int) -> Tuple[str, int]:
        # binary search over offsets
        lo, hi = 0, len(self.offs)-1
        while lo <= hi:
            mid = (lo+hi)//2
            if mid == len(self.offs)-1 or self.offs[mid+1] > idx >= self.offs[mid]:
                return self.paths[mid], idx - self.offs[mid]
            if idx < self.offs[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        raise IndexError(idx)

    def __getitem__(self, idx: int):
        path, local = self._locate(idx)
        with h5py.File(path, "r") as f:
            H = f[self.hits_ds][local]  # (512, F)
            y = int(f[self.label_group][local, self.label_col])

        # drop padded rows
        valid = ~(np.all(H == 0.0, axis=1))
        H = H[valid]
        if H.shape[0] == 0:
            # degenerate: return empty volume with zero label (rare)
            x = np.zeros((self.T, 2, self.nx, self.ny, self.nz), dtype=self.dtype)
            return torch.from_numpy(x), y

        t = H[:, self.tcol].astype(np.float64, copy=False)
        x = H[:, self.xcol].astype(np.float64, copy=False)
        yv = H[:, self.ycol].astype(np.float64, copy=False)
        z = H[:, self.zcol].astype(np.float64, copy=False)

        # event-relative time
        t0 = np.min(t)
        t_rel = t - t0

        # time bins
        tb = np.floor((t_rel - self.tmin) / self.dt).astype(np.int64)
        # spatial indices
        ix = np.floor((x - self.xmin) / self.vx).astype(np.int64)
        iy = np.floor((yv - self.ymin) / self.vy).astype(np.int64)
        iz = np.floor((z - self.zmin) / self.vz).astype(np.int64)

        # mask to keep only in-range bins/voxels
        m = (tb >= 0) & (tb < self.T) & \
            (ix >= 0) & (ix < self.nx) & \
            (iy >= 0) & (iy < self.ny) & \
            (iz >= 0) & (iz < self.nz)

        if not np.any(m):
            vol = np.zeros((self.T, 2, self.nx, self.ny, self.nz), dtype=self.dtype)
            return torch.from_numpy(vol), y

        tb, ix, iy, iz = tb[m], ix[m], iy[m], iz[m]

        # accumulate counts
        counts = np.zeros((self.T, self.nx, self.ny, self.nz), dtype=np.float32)
        np.add.at(counts, (tb, ix, iy, iz), 1.0)
        occ = (counts > 0).astype(np.float32)

        vol = np.stack([occ, counts], axis=1)   # [T, C=2, nx, ny, nz]
        vol = vol.astype(self.dtype, copy=False)
        return torch.from_numpy(vol), y